# Lesson 12 - Keskusteluhistorian tiivistäminen Agentin luonnospaperilla

Tässä muistikirjassa osoitetaan, miten hallitaan kontekstia pitkissä keskusteluissa Microsoft Agent Frameworkin avulla. Kun keskustelut kasvavat, tokenien määrä kasvaa — lopulta ylittäen mallin konteksti-ikkunan koon. Ratkaisemme tämän **kontekstin tiivistämismallilla** ja **agentin luonnospaperilla** jatkuvan muistin varmistamiseksi.

## Mitä opit:
1. **Miksi kontekstin hallinta on tärkeää**: Token-rajoitusten ja konteksti-ikkunoiden ymmärtäminen
2. **Kontekstia ymmärtävät agentit**: Agenttien rakentaminen, jotka hallitsevat omaa keskustelukontekstiaan
3. **Kontekstin tiivistämismalli**: Työkalujen käyttö keskusteluhistorian tiivistämiseen
4. **Agentin luonnospaperi**: Jatkuva muisti, joka säilyy kontekstin tiivistämisestä huolimatta

## Edellytykset:
- Azure OpenAI -ympäristö, jossa ympäristömuuttujat on määritetty
- Perustiedot agenteista aiemmista oppitunneista


## Asennus


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Miksi kontekstinhallinta on tärkeää

Jokaisella LLM:llä on rajallinen **kontekstikehys** — maksimimäärä tokeneita, joita se voi käsitellä yhdessä pyynnössä. Monivaiheisen keskustelun edetessä:

- **Tokenien määrä kasvaa lineaarisesti** jokaisen käyttäjän viestin ja avustajan vastauksen myötä.
- **Prompt-tokenit ovat suurin kustannuserä**, koska koko historia lähetetään uudelleen jokaisella kierroksella.
- Lopulta keskustelu **ylittää kontekstikehyksen**, jolloin malli joko katkaisee tai aiheuttaa virheen.

### Strategiat kontekstin hallintaan

| Strategia | Miten se toimii | Kompromissi |
|---|---|---|
| **Katkaisu** | Pudottaa vanhimmat viestit | Menettää alkuperäisen kontekstin |
| **Tiivistäminen** | Tiivistää vanhemmat viestit yhteenvedoksi | Jotain yksityiskohtia katoaa, mutta avainkohdat säilyvät |
| **Muisti / Ulkoinen muisti** | Tallentaa keskeiset tiedot keskustelun ulkopuolelle | Vaatii työkalukutsuja, mutta säilyy kaikissa lyhennyksissä |

Tässä muistikirjassa yhdistämme **tiivistämisen** ja **muistityökalun**, jotta toimija voi säilyttää jatkuvuuden, vaikka keskusteluhistoria tiivistetään.


## Kontekstin Tietoinen Agentin Luominen


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Pitkän Keskustelun Simulointi

Käydään läpi monivaiheinen keskustelu nähdäksemme, miten konteksti kertyy. Agentin tulisi säilyttää keskeiset tiedot (mieltymykset, budjetti, matkustuspäivät) eri vaiheiden välillä ja osoittaa jatkuvuutta.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Huomaa, miten agentti säilyttää kontekstin aiemmista vuoroista — se tietää Japanista, sushista, temppeleistä, valokuvauksesta, 3000 dollarin budjetista, yksinmatkailusta ja huhtikuun aikataulusta. Lyhyessä keskustelussa tämä toimii hyvin, mutta keskustelun kasvaessa koko historiankin uudelleen lähettäminen käy kalliiksi.

Jatketaan keskustelua useammalla vuorolla nähdäksesi, miten konteksti kertyy:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Yhteenvetokuvio kontekstista

Keskustelun kasvaessa voimme käyttää **yhteenvetotyökalua** tiivistämään kertynyttä kontekstia kompaktiksi muodoksi. Agentti kutsuu tätä työkalua tallentaakseen keskeiset mieltymykset, jotta olennaiset tiedot säilyvät, vaikka vanhemmat viestit poistettaisiin.

Tämä kuvio on perustana kehittyneemmille historian supistamiskeinoille:
1. Agentti tunnistaa keskustelusta keskeiset faktat
2. Se kutsuu yhteenvetotyökalua tallentaakseen ne
3. Vanhempia viestejä voidaan turvallisesti poistaa, koska yhteenveto kattaa tärkeät asiat

Alla määrittelemme `summarize_preferences`-työkalun, jota agentti voi kutsua tallentaakseen oppimansa tiiviin yhteenvedon.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Yhteenveto

Tässä oppitunnissa opit hallitsemaan kontekstia pitkäkestoisissa agenttikeskusteluissa Microsoft Agent Frameworkin avulla:

### Keskeiset käsitteet
- **Kontekstin ikkunoilla on rajat** — jokainen keskusteluhistorian token maksaa ja lasketaan mukaan rajaan.
- **Tiivistystyökalut** antavat agentille mahdollisuuden tiivistää kertynyt konteksti kompakteiksi yhteenvetoiksi, vähentäen tokenien käyttöä samalla kun tärkeä tieto säilyy.
- **Agentin muistiinpanot** tarjoavat pysyvän ulkoisen muistin, joka säilyy keskustelun lyhentämisestä huolimatta.

### Mitä rakensit
- **Kontekstia ymmärtävä agentti**, joka ylläpitää jatkuvuutta monivuoropuhelussa
- **Tiivistystyökalu** (`summarize_preferences`), joka tallentaa olennaiset käyttäjätiedot kompaktissa muodossa
- **Monivuoropuhelu**, joka osoittaa kontekstin säilyttämisen ja muutosten käsittelyn

### Käytännön sovellukset
- **Asiakaspalvelubotit**: muistavat mieltymykset pitkien tukikeskustelujen ajan
- **Henkilökohtaiset avustajat**: seuraavat käynnissä olevia projekteja ilman kontekstin uudelleen selittämistä
- **Koulutustutorit**: ylläpitävät opiskelijan edistymistä useiden vuorovaikutusten ajan

### Seuraavat askeleet
- Toteuta täydellinen muistiinpanotyökalu tiedostopohjaisella pysyvyydellä
- Lisää automaattinen historiatietojen karsiminen tiivistämisen jälkeen
- Yhdistä vektoritietokantoihin semanttisen muistin hakua varten
- Rakenna agenteja, jotka voivat jatkaa keskusteluja päiviä myöhemmin täydellisellä kontekstilla


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty käyttämällä tekoälypohjaista käännöspalvelua [Co-op Translator](https://github.com/Azure/co-op-translator). Pyrimme tarkkuuteen, mutta otathan huomioon, että automaattiset käännökset voivat sisältää virheitä tai epätarkkuuksia. Alkuperäistä asiakirjaa sen alkuperäisellä kielellä tulee pitää auktoritatiivisena lähteenä. Tärkeiden tietojen osalta suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinymmärryksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
